# [Sensitivity Analysis] The Topographical Underfitting (KAN-TabNet) | Step LR | Forest Cover

**Optimized Reference Configuration:** `n_d=20, n_a=20`, `kan_grid_size=5`, `kan_spline_order=3`, `initial_lr=0.002`

### Methodological Context: The Control Variables
To precisely isolate the effects of spline resolution on geospatial data, this sensitivity analysis uses the exact same massive routing pipeline (`20x20`), cubic spline order (`k=3`), StepLR schedule, and initial learning rate (`0.002`) as our optimized reference configuration.

By fixing the structural routing capacity and thermodynamic optimization environment, we strictly guarantee that any performance variations observed in this notebook are purely the result of restricting the internal grid intervals ($G$) of the KAN layers.

### The Hypothesis
In this study, we evaluate a highly constrained grid resolution (`kan_grid_size=3`). Experiments on the Higgs Boson dataset demonstrated that $G=3$ acts as an excellent structural regularizer against smooth, continuous quantum noise. However, the Forest Cover dataset presents a drastically different physical reality: geography is jagged, non-linear, and heavily reliant on 54 features containing highly sparse, one-hot encoded categorical data (e.g., 40 distinct soil types and 4 wilderness areas).

We hypothesize that geospatial topography requires higher mathematical flexibility to accurately map sudden, sharp boundaries between different environmental classes. By restricting the grid size from 5 to 3, we artificially constrain the B-splines, preventing them from molding to these jagged categorical distributions. This notebook serves to empirically evaluate whether this restriction causes "topographical underfitting," actively harming the model's ability to classify sudden ecological shifts compared to our standard $G=5$ reference configuration.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier

dataset = fetch_covtype()

X = dataset.data
# CovType is 1-indexed (1 to 7); PyTorch expects 0-indexed labels
y = dataset.target - 1

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (464809, 54)
Valid shape: (58101, 54)
Test shape:  (58102, 54)


### Model Training

In [5]:
MAX_EPOCHS = 1000

In [6]:
# Hyperparameters from original paper.
# Notes:
# - momentum is 1 - 0.7 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 18 epochs approximates the paper's 500 iterations
#   (based on a batch size of 16384 and ~464k training samples).
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-4,
    'momentum': 0.3,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=18, gamma=0.95),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=20,
    n_a=20,
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.002),
    use_kan=True,
    kan_grid_size=3,
    kan_spline_order=3,
    seed=seed,
    verbose=50
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=100,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.62603 | valid_accuracy: 0.26741 |  0:00:09s
epoch 50 | loss: 0.28251 | valid_accuracy: 0.89422 |  0:06:56s
epoch 100| loss: 0.17356 | valid_accuracy: 0.93499 |  0:13:45s
epoch 150| loss: 0.13779 | valid_accuracy: 0.94766 |  0:20:33s
epoch 200| loss: 0.12962 | valid_accuracy: 0.95031 |  0:27:27s
epoch 250| loss: 0.10955 | valid_accuracy: 0.95663 |  0:34:22s
epoch 300| loss: 0.10215 | valid_accuracy: 0.95964 |  0:41:15s
epoch 350| loss: 0.09215 | valid_accuracy: 0.96114 |  0:48:08s
epoch 400| loss: 0.08821 | valid_accuracy: 0.96207 |  0:55:01s
epoch 450| loss: 0.07931 | valid_accuracy: 0.96322 |  1:01:51s
epoch 500| loss: 0.07758 | valid_accuracy: 0.96429 |  1:08:42s
epoch 550| loss: 0.07317 | valid_accuracy: 0.96539 |  1:15:32s
epoch 600| loss: 0.07247 | valid_accuracy: 0.96666 |  1:22:22s
epoch 650| loss: 0.07425 | valid_accuracy: 0.96542 |  1:29:16s
epoch 700| loss: 0.06879 | valid_accuracy: 0.96534 |  1:36:09s
epoch 750| loss: 0.06807 | valid_accuracy: 0.96666 |  1

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 377,496


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.96766


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/forest_cover/tables'
MODELS_DIR = f'{BASE_DIR}/results/forest_cover/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Forest Cover",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/05_kan_sensitivity_grid_3_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/05_kan_sensitivity_grid_3_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/forest_cover/tables/05_kan_sensitivity_grid_3_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/forest_cover/models/05_kan_sensitivity_grid_3_377k.zip
